In [1]:
import sys
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm

In [2]:
sys.path.append("/home/alexis/Documents/COMP558/COMP558-FinalProject/code/")
# sys.path.append("/workspaces/python-opencv/repo/code/")

In [3]:
from optical_flow.lucas_kanade import LucasKanade
from feature_detection.shi_tomasi import ShiTomasiDetector
from optical_flow.bounding_box import BoundingBox
from optical_flow.points_utils import subset_points 
from optical_flow.ess import ess_search
from optical_flow.ess import FuzzyBoundingBox
from optical_flow.ess import get_largest_bounding_box

In [4]:
video_name = "/home/alexis/Documents/master/vision/repo/libfreenect/wrappers/python/out/VIDEO-20250113-190349.mp4"

output_name = "out/lk_ess_cleanup.mp4"

In [5]:
x, y, w, h = 275, 200, 110, 85

In [6]:
shi_tomasi_params = {"params" : {"maxCorners" : 1000, "qualityLevel" : 0.01, "minDistance" : 5, "blockSize" : 5}}
lucas_kanade_params = {}

In [7]:
weight_pts = 1
weight_delta = 1

In [8]:
shi_tomasi = ShiTomasiDetector(**shi_tomasi_params)
lucas_kanade = LucasKanade(**lucas_kanade_params)
initial_bbox = BoundingBox(x, y, w, h)

In [9]:
cap = cv.VideoCapture(video_name)
fps = cap.get(cv.CAP_PROP_FPS)

ret, frame = cap.read()

fourcc = cv.VideoWriter_fourcc(*'mp4v')
writer = cv.VideoWriter(output_name, fourcc, fps, frame.shape[:-1][::-1])

In [10]:
points = shi_tomasi.detect_features(frame)
points = subset_points(points, initial_bbox)

prev_frame = frame.copy()
prev_bbox = initial_bbox

full_frame_bbox = BoundingBox(0, 0, frame.shape[1], frame.shape[0])

In [11]:
while ret:

    count, error, old_points, new_points = lucas_kanade.track_frame(prev_frame, frame, points)

    def ess_search_function(bbox : FuzzyBoundingBox) -> float:
        
        large = get_largest_bounding_box(bbox)
        
        points_in = subset_points(points, large)
        points_sum = len(points_in)

        # FIT TIGHTER
        # left
        diff_l = np.abs(points[:, 0] -  bbox.l.mid_point)
        delta_l = np.clip(np.min(diff_l) - bbox.l.span/2, 0, np.inf)

        # right
        diff_r = np.abs(points[:, 0] -  bbox.r.mid_point)
        delta_r = np.clip(np.min(diff_r) - bbox.r.span/2, 0, np.inf)

        # bottom
        diff_b = np.abs(points[:, 1] -  bbox.b.mid_point)
        delta_b = np.clip(np.min(diff_b) - bbox.b.span/2, 0, np.inf)

        # top
        diff_t = np.abs(points[:, 1] -  bbox.t.mid_point)
        delta_t = np.clip(np.min(diff_t) - bbox.t.span/2, 0, np.inf)

        return weight_pts * points_sum - weight_delta * (delta_l + delta_r + delta_b + delta_t)
    
    bbox, error = ess_search(full_frame_bbox, ess_search_function)

    img2 = frame.copy()
    for pt in points:
        x, y = pt.ravel()
        img2 = cv.circle(img2, (int(x), int(y)), 5, (255, 0, 0), -1)

    img2 = cv.rectangle(img2, (bbox.x, bbox.y), (bbox.x + bbox.w, bbox.y + bbox.h), 255, 2)

    writer.write(img2)

    prev_bbox = bbox
    points = new_points.copy()
    prev_frame = frame.copy()
    ret, frame = cap.read()

In [12]:
cap.release()
writer.release()